# get similarity convergence

In [30]:
import os
import platform

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

In [31]:
os_system = platform.system() # 맥북은 Darwin, 윈도우는 Windows

# 현재 프로젝트 폴더 위치 지정. os.getcwd()는 지금 코드 실행하는 현 위치를 출력해줍니다.
study1_dir = os.getcwd()

# data/processed 폴더 위치 지정
processed_data_dir = study1_dir + ('\\data\\processed\\' if os_system == 'Windows' else '/data/processed/')

# graph 이미지 저장할 폴더 위치 지정
graph_image_dir = study1_dir + ('\\graph\\slope' if os_system == 'Windows' else '/graph/slope')

In [32]:
# 폴더 없으면 생성
os.makedirs(graph_image_dir, exist_ok=True)

In [33]:
# 유사도, 유사도평균 coherence값을 저장한 테이블 읽어오기
tbl_data = pd.read_csv(processed_data_dir + 'similarity_coherence.csv', index_col=0, keep_default_na=False)
tbl_data[0:3]

,similarity_key1_money,similarity_key2_money,similarity_key3_money,similarity_key4_money,similarity_key5_money,similarity_key6_money,similarity_key7_money,similarity_key8_money,similarity_key9_money,similarity_key10_money,...,similarity_friend29_friend,similarity_friend30_friend,coherence_key_money,coherence_money_money,coherence_friend_money,coherence_money,coherence_key_friend,coherence_money_friend,coherence_friend_friend,coherence_friend
subject,,,,,,,,,,,,,,,,,,,,,
1,0.198052,0.261321,1.000000,0.098472,0.035303,-0.023307,0.032843,0.03683750296324395,0.031676,0.08602895313983461,...,0.122040,0.012896,0.142339,0.108646,0.066801,0.105929,0.111955,0.109923,0.079585,0.100488
2,0.105224,0.113665,0.238537,0.194226,0.025999,0.128698,0.072546,0.04600394279655973,0.044098,0.12166701755294584,...,0.043440,0.266920,0.091669,0.127313,0.087375,0.102119,0.092107,0.098032,0.121426,0.103855
3,0.113665,0.091659,0.086067,0.070577,0.064165,0.116383,0.122404,0.1431124424389889,0.098101,0.019974692332244803,...,0.049411,-0.006704,0.122393,0.159426,0.091905,0.124574,0.180068,0.118613,0.113313,0.137331


In [34]:
def get_smoothed_similarity_func(similarity_df: pd.DataFrame, i_subject: int, topic: str):
    # trial 번호와 해당 trial에 대한 similarity 값을 배열로 변환
    trial_numbers = np.array(list(range(1, 31)))

    # 특정 피험자의 유사도 점수들 받아오기
    similarity_values = similarity_df.iloc[i_subject].tolist()
    similarity_values = np.array([float(value) if value != '' else 0.0 for value in similarity_values])

    # 데이터를 보간하는 함수 생성
    interpolation_function = interp1d(trial_numbers, similarity_values, kind='quadratic')

    # 정수값에 대한 데이터 추출
    integer_trial_numbers = np.arange(1, 31)
    integer_similarity_values = interpolation_function(integer_trial_numbers)

    # 부드러운 곡선을 위해 trial 번호를 더 자세히 나누기
    fine_trial_numbers = np.linspace(1, 30, 300)
    smoothed_similarity_values = interpolation_function(fine_trial_numbers)

    # 1차 함수 (선형 회귀)를 생성하여 예측값 얻기
    z = np.polyfit(integer_trial_numbers, integer_similarity_values, 1)
    p = np.poly1d(z)
    predicted_values = p(integer_trial_numbers)

    # 기울기와 절편 구하기
    slope = z[0]
    intercept = z[1]

    # 그래프 저장할 위치
    graph_image_path = (f'\\graph\\{topic}\\Subject_{i_subject}_similarity_curve.png' if os_system == 'Windows' else f'{graph_image_dir}/{topic}/Subject_{i_subject}_similarity_curve.png')
    # 폴더 없으면 생성
    os.makedirs(f'\\graph\\{topic}' if os_system == 'Windows' else f'{graph_image_dir}/{topic}', exist_ok=True)
    
    # 그래프 생성
    plt.figure(figsize=(10, 5))
    plt.plot(integer_trial_numbers, predicted_values, label='Linear Regression', color='g', linestyle='-')
    plt.plot(fine_trial_numbers, smoothed_similarity_values, label='Smoothed similarities', color='r')
    plt.scatter(integer_trial_numbers, integer_similarity_values, label='Real similarity Data', marker='o', color='b')
    plt.xlabel('Trial')
    plt.ylabel('Similarity Value')
    plt.title(f'Subject {i_subject}: Smoothed Similarity Curve')
    plt.legend()
    plt.grid(True)
    plt.savefig(graph_image_path)
    # plt.show()

    # 그래프 표시하지 않음
    plt.close()

    return predicted_values, (slope, intercept)


In [35]:
def add_convergence_values_to_df(start_index: int,end_index: int, dataframe: pd.DataFrame, seed_word: str, target_word: str):
    similarity_seed_target = tbl_data.iloc[:, start_index:end_index]
    slopes = []
    intercepts = []
    for i_subject in range(len(similarity_seed_target)):
        predicted_values, (slope, intercept) = get_smoothed_similarity_func(similarity_df = similarity_seed_target,
                                                                            i_subject = i_subject,
                                                                            topic = f'{seed_word}_{target_word}')
        slopes.append(slope)
        intercepts.append(intercept)
        
    # 모든 피험자의 convergence값을 얻은 후,
    tbl_data[f'convergence_slope_{seed_word}_{target_word}'] = slopes
    tbl_data[f'convergence_intercept_{seed_word}_{target_word}'] = intercepts

### target: money

In [36]:
target_word = 'money'

In [37]:

add_convergence_values_to_df(start_index=0,
                             end_index=30,
                             dataframe=tbl_data,
                             seed_word='key',
                             target_word=target_word)
add_convergence_values_to_df(start_index=30,
                             end_index=60,
                             dataframe=tbl_data,
                             seed_word='money',
                             target_word=target_word)
add_convergence_values_to_df(start_index=60,
                             end_index=90,
                             dataframe=tbl_data,
                             seed_word='friend',
                             target_word=target_word)

### target: friend

In [38]:
target_word = 'friend'

In [39]:
add_convergence_values_to_df(start_index=90,
                             end_index=120,
                             dataframe=tbl_data,
                             seed_word='key',
                             target_word=target_word)
add_convergence_values_to_df(start_index=120,
                             end_index=150,
                             dataframe=tbl_data,
                             seed_word='money',
                             target_word=target_word)
add_convergence_values_to_df(start_index=150,
                             end_index=180,
                             dataframe=tbl_data,
                             seed_word='friend',
                             target_word=target_word)

## save csv

In [40]:
tbl_data[0:3]

,similarity_key1_money,similarity_key2_money,similarity_key3_money,similarity_key4_money,similarity_key5_money,similarity_key6_money,similarity_key7_money,similarity_key8_money,similarity_key9_money,similarity_key10_money,...,convergence_slope_money_money,convergence_intercept_money_money,convergence_slope_friend_money,convergence_intercept_friend_money,convergence_slope_key_friend,convergence_intercept_key_friend,convergence_slope_money_friend,convergence_intercept_money_friend,convergence_slope_friend_friend,convergence_intercept_friend_friend
subject,,,,,,,,,,,,,,,,,,,,,
1,0.198052,0.261321,1.000000,0.098472,0.035303,-0.023307,0.032843,0.03683750296324395,0.031676,0.08602895313983461,...,-0.006637,0.211517,-0.000705,0.075498,0.002484,0.073455,0.003076,0.062250,-0.002541,0.116312
2,0.105224,0.113665,0.238537,0.194226,0.025999,0.128698,0.072546,0.04600394279655973,0.044098,0.12166701755294584,...,-0.005425,0.211397,-0.001220,0.106292,-0.001054,0.108437,0.003742,0.040030,-0.004926,0.197782
3,0.113665,0.091659,0.086067,0.070577,0.064165,0.116383,0.122404,0.1431124424389889,0.098101,0.019974692332244803,...,0.004671,0.087023,-0.002327,0.127966,0.002964,0.134127,-0.001353,0.139587,-0.008335,0.242504


In [41]:
# 단어 있는 버전 csv 저장
tbl_data.to_csv(processed_data_dir + 'convergence.csv')
